# ActiveSAM 单图演示 · Google Colab

选择 GPU 运行时，按顺序运行前 6 个代码格。首次运行需要授权挂载 Google Drive。权重和数据默认读取你在 Drive 中的 `models/sam3` 和 `datasets/ADE20K`，每次测试归档到 `results/ActiveSAM`。

笔记本复用 Colab 的 PyTorch 和 torchvision，使用 pip 安装部署目录中的预编译依赖。GPU 缺少原生 BF16 时会提示警告，并用 FP32 尝试推理。FP32 的显存用量可能更高。

每张图都会展示分割结果、叠加图、像素分数热图和类别统计。归档还保存原始标签、完整分数图、参数和运行数据。

[ActiveSAM v2](https://arxiv.org/html/2606.16996v2) · [源码版本](https://github.com/xheihex/ActiveSAM/tree/colab-deployment/colab_deployment)


## 1. 挂载 Drive 并设置路径

ADE20K 文件夹应包含 `images/validation`。有对应的 `annotations/validation` 时，会自动计算单图指标。也可切换到 VOC21 并改成它的数据目录。


In [ ]:
# @title 挂载 Drive 并设置路径
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

WEIGHTS_PATH = "/content/drive/MyDrive/models/sam3/sam3.pt" # @param {type:"string"}
DATASET_PATH = "/content/drive/MyDrive/datasets/ADE20K"  # @param {type:"string"}
OUTPUT_ROOT = "/content/drive/MyDrive/results/ActiveSAM"  # @param {type:"string"}

WEIGHTS_PATH = Path(WEIGHTS_PATH)
DATASET_PATH = Path(DATASET_PATH)
OUTPUT_ROOT = Path(OUTPUT_ROOT)

DATASET_PRESET = "ade20k"  # 或 "voc21"，同时填写对应的数据目录
MAX_INPUT_SIDE = 1024     # 0 使用原始尺寸
BUCKET_SIZE = 32          # 每组一起处理的文本提示数
OVERLAY_ALPHA = 0.5
WORDNET_DIR = Path("/content/drive/MyDrive/datasets/nltk_data")  # 可选

if not WEIGHTS_PATH.is_file():
    raise FileNotFoundError(f"找不到 SAM 3 权重：{WEIGHTS_PATH}")
if not DATASET_PATH.is_dir():
    raise FileNotFoundError(f"找不到解压后的数据集：{DATASET_PATH}")


## 2. 下载源码

默认拉取个人分叉的 Colab 部署分支。每次运行都会创建新的部署目录，分别存放仓库和日志。单图适配与依赖清单都在仓库的 colab_deployment 目录。


In [ ]:
# @title 下载 GitHub 代码
import json
import os
import subprocess
import uuid

GITHUB_URL = "https://github.com/xheihex/ActiveSAM.git" # @param {type:"string"}
GIT_REF = "colab-deployment" # @param {type:"string"}


def checkout_github(github_url, git_ref, repo, logs):
    logs.mkdir(parents=True)
    log_path = logs / "checkout.log"
    git_env = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1", GIT_TERMINAL_PROMPT="0")
    commands = [
        ["git", "init", "--quiet", str(repo)],
        ["git", "-C", str(repo), "fetch", "--depth=1", "--", github_url, git_ref],
        ["git", "-C", str(repo), "checkout", "--quiet", "--detach", "FETCH_HEAD"],
    ]
    try:
        with log_path.open("w", encoding="utf-8") as log:
            for command in commands:
                subprocess.run(command, check=True, env=git_env, stdout=log, stderr=subprocess.STDOUT)
    except subprocess.CalledProcessError:
        print(log_path.read_text(encoding="utf-8", errors="replace")[-3000:])
        raise RuntimeError(f"下载源码失败，日志见 {log_path}") from None
    commit = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
    return repo, {"github_url": github_url, "git_ref": git_ref, "git_commit": commit, "checkout_log": str(log_path)}


DEPLOYMENT_ID = uuid.uuid4().hex[:12]
DEPLOYMENT_DIR = Path("/content/activesam-deployments") / DEPLOYMENT_ID
LOGS = DEPLOYMENT_DIR / "logs"
print(f"正在下载 {GITHUB_URL} 的 {GIT_REF}，日志见 {LOGS / 'checkout.log'}", flush=True)
REPO_DIR, SOURCE_INFO = checkout_github(GITHUB_URL, GIT_REF, DEPLOYMENT_DIR / "ActiveSAM", LOGS)
SOURCE_INFO["deployment_id"] = DEPLOYMENT_ID
(LOGS / "source_info.json").write_text(json.dumps(SOURCE_INFO, ensure_ascii=False, indent=2), encoding="utf-8")

DEPLOYMENT_CODE = REPO_DIR / "colab_deployment"
if not (DEPLOYMENT_CODE / "adapter.py").is_file():
    raise FileNotFoundError("此 Git 引用缺少单图部署目录，请选择分叉的 colab-deployment 分支。")
print("源码", REPO_DIR, SOURCE_INFO["git_commit"][:8])


## 3. 安装单图所需的包

使用 pip 安装部署清单中的预编译包，并复用 Colab 自带的 PyTorch 和 torchvision。


In [ ]:
import sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--only-binary=:all:",
        "-r", str(DEPLOYMENT_CODE / "requirements.txt"),
    ],
    check=True,
)


## 4. 加载模型

首次读取时会显示从 Drive 复制的进度、速度和预计剩余时间，同时计算文件指纹。同一会话里再次加载同一个权重时，会提示复用本地副本。

模型加载按完成的阶段推进，依次显示构建模型、读取并载入权重、移到 GPU、初始化类别提示与签名。每个阶段用时可能不同。


In [ ]:
import csv
from datetime import datetime
import gc
import hashlib
import importlib.metadata
import json
import platform
import runpy
import random
import shutil
import time
import traceback
import uuid
from zoneinfo import ZoneInfo

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import nltk
from PIL import Image
import torch
from IPython.display import display, Markdown
from tqdm.auto import tqdm

if globals().get("IMPORTED_REPO_DIR") != REPO_DIR:
    if "model" in globals():
        del model
        gc.collect()
        torch.cuda.empty_cache()
    old_repo = globals().get("IMPORTED_REPO_DIR")
    if old_repo and str(old_repo) in sys.path:
        sys.path.remove(str(old_repo))
    sys.path.insert(0, str(REPO_DIR))
    for name in tuple(sys.modules):
        if name.split(".", 1)[0] in {"colab_deployment", "sam3"}:
            del sys.modules[name]

from colab_deployment.adapter import build_model, canonicalize, load_vocabulary
IMPORTED_REPO_DIR = REPO_DIR

local_wordnet = Path("/content/nltk_data")
nltk.data.path[:0] = [str(WORDNET_DIR), str(local_wordnet)]
try:
    nltk.corpus.wordnet.synsets("bicycle")
except LookupError:
    nltk.download("wordnet", download_dir=str(local_wordnet), raise_on_error=True)


config_file = REPO_DIR / "configs" / f"cfg_{DATASET_PRESET}.py"
model_cfg = {
    **runpy.run_path(str(REPO_DIR / "configs/base_config.py"))["model"],
    **runpy.run_path(str(config_file))["model"],
}
model_cfg.pop("type")
vocab_path = REPO_DIR / model_cfg["vocabulary"]
signature_path = REPO_DIR / model_cfg["signatures"]
prompt_names, prompt_classes = load_vocabulary(str(vocab_path))
class_count = max(prompt_classes) + 1
CLASS_INFO = []
for class_id in range(class_count):
    aliases = [name for name, index in zip(prompt_names, prompt_classes) if index == class_id]
    CLASS_INFO.append({
        "class_id": class_id, "name": aliases[0], "aliases": aliases,
        "canonicalized_aliases": [canonicalize(name) for name in aliases],
    })

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


local_weight_dir = Path("/content/activesam_weights")
local_weight_dir.mkdir(exist_ok=True)
local_checkpoint = local_weight_dir / "sam3.pt"
partial_weight = local_weight_dir / "sam3.pt.partial"

source_stat = WEIGHTS_PATH.stat()
weight_key = (str(WEIGHTS_PATH.resolve()), source_stat.st_size, source_stat.st_mtime_ns)
cached_weight = globals().get("WEIGHT_CACHE")
if cached_weight and cached_weight["key"] == weight_key and local_checkpoint.is_file() and (
    local_checkpoint.stat().st_size == source_stat.st_size
):
    WEIGHT_SHA256 = cached_weight["sha256"]
    print("复用 Colab 本地的权重副本。", flush=True)
else:
    digest = hashlib.sha256()
    with (
        WEIGHTS_PATH.open("rb") as source,
        partial_weight.open("wb") as target,
        tqdm(total=source_stat.st_size, desc="从 Drive 复制权重", unit="B", unit_scale=True, unit_divisor=1024) as copy_progress,
    ):
        for chunk in iter(lambda: source.read(4 * 1024 * 1024), b""):
            target.write(chunk)
            digest.update(chunk)
            copy_progress.update(len(chunk))
    current_stat = WEIGHTS_PATH.stat()
    if (current_stat.st_size, current_stat.st_mtime_ns) != weight_key[1:] or (
        partial_weight.stat().st_size != source_stat.st_size
    ):
        raise RuntimeError("权重在复制时发生变化，请等上传完成再重跑。")
    WEIGHT_SHA256 = digest.hexdigest()
    os.replace(partial_weight, local_checkpoint)
    WEIGHT_CACHE = {"key": weight_key, "sha256": WEIGHT_SHA256}


model_cfg.update(
    vocabulary=str(vocab_path),
    signatures=str(signature_path),
    sam3_checkpoint=str(local_checkpoint),
    bucket_size=BUCKET_SIZE,
)
if "model" in globals():
    del model
gc.collect()
torch.cuda.empty_cache()
model, NATIVE_BF16 = build_model(model_cfg)
assert model.num_classes == class_count
LOADED_PRESET = DATASET_PRESET
LOADED_REPO_DIR = REPO_DIR
LOADED_WEIGHT_SOURCE = str(WEIGHTS_PATH.resolve())
LOADED_WEIGHT_SHA256 = WEIGHT_SHA256
LOADED_MODEL_CONFIG = json.loads(json.dumps(model_cfg))
VERSION_INFO = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(),
    "native_bf16": NATIVE_BF16,
    "sam3_compute_mode": "bf16_autocast" if NATIVE_BF16 else "fp32",
    "integration": "standalone_single_image",
    "packages": {
        name: importlib.metadata.version(name)
        for name in ["torchvision", "numpy", "Pillow", "nltk", "matplotlib", "timm", "iopath"]
    },
}
print("模型加载完成。权重 SHA-256", WEIGHT_SHA256)


## 5. 准备可视化和结果归档
每次测试都会创建独立的「北京时间 + 随机后缀」目录。先在 Colab 本地保存，再逐文件复制并验证 SHA-256，最后写入 `COMPLETE.json`。完整目录中包含以下结果。

| 文件 | 内容 |
| --- | --- |
| `input_original.png`、`input_used.png` | 原始像素的 RGB 图和实际推理图，颜色方向与源码读取方式一致 |
| `labels.png`、`labels.npy` | 每个像素的原始类别 ID，可直接读取计算 |
| `scores.npz` | 所有类别的 FP32 分数图、出现分数、预览准入类别、最终标签对应分数、最大分数和逐像素最高分标签 |
| `class_results.csv`、`result.json` | 各类分数统计、像素覆盖、参数、文件指纹、版本、耗时、峰值显存与可选单图指标 |
| `segmentation_color.png`、`overlay.png`、`visualization.png` | 类别着色、透明叠加、带图例和分数热图的综合图 |
| `vocabulary.txt`、`signatures.npz` | 本次使用的类别顺序、别名和 ECD 签名 |
| `ground_truth_*.png` | 有标注时，保存原始标注和映射到模型类别 ID 后的标注 |
| `COMPLETE.json` | 归档成功标记与文件校验值，写入顺序为最后一个 |

**分数如何理解**。`class_scores` 是原版融合并按出现分数加权后的模型响应，各类别之间没有归一化。可以用它观察置信程度。ECD 可能把像素分给分数不是最高的类别，因此另存 `assigned_class_score`（最终标签对应的分数）和 `max_class_score`（最高分）。这些响应值不能直接解释成“正确概率”。

`presence_scores` 按类别别名取最大值。预览未准入的类为零，第二轮筛掉的提示保留首层分数，完成掩膜解码的提示保留完整解码分数。`preview_admitted_class_ids` 记录的是第一轮准入类别。`predicted_class_ids` 则记录最终标签里实际出现的类别。

`confidence_threshold=0.3` 是作者对**实例分支**的过滤阈值。单图标签和语义分支仍由原版流程处理。完整 FP32 分数图默认全部保存，较大图片的文件和写盘时间也会增加。


In [ ]:
def write_json(path, value):
    Path(path).write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8"
    )


def summarize_arrays(labels, class_scores, presence_scores, active_ids, class_info, ground_truth=None):
    count = len(class_info)
    if class_scores.dtype != np.float32 or class_scores.shape != (count, *labels.shape):
        raise ValueError("分数图类型或尺寸与标签不匹配。")
    if not np.isfinite(class_scores).all() or not np.isfinite(presence_scores).all():
        raise ValueError("模型分数含非有限数值。")
    if labels.min() < 0 or labels.max() >= count:
        raise ValueError("标签含词表范围之外的类别 ID。")
    assigned = np.take_along_axis(class_scores, labels[None], axis=0)[0]
    maximum = class_scores.max(axis=0)
    raw_argmax = class_scores.argmax(axis=0).astype(np.uint8)
    rows = []
    for item in class_info:
        class_id = item["class_id"]
        chosen = labels == class_id
        pixels = int(chosen.sum())
        row = {
            **item,
            "presence_score": float(presence_scores[class_id]),
            "preview_admitted": bool(class_id in active_ids),
            "has_nonzero_score_map": bool(class_scores[class_id].max() > 0),
            "pixel_count": pixels,
            "pixel_fraction": pixels / labels.size,
            "score_map_max": float(class_scores[class_id].max()),
            "assigned_score_mean": float(assigned[chosen].mean()) if pixels else None,
            "assigned_score_min": float(assigned[chosen].min()) if pixels else None,
            "assigned_score_max": float(assigned[chosen].max()) if pixels else None,
        }
        rows.append(row)
    metrics = None
    if ground_truth is not None:
        if ground_truth.shape != labels.shape:
            raise ValueError("人工标注尺寸与预测不同。")
        valid = ground_truth != 255
        if valid.any():
            if ground_truth[valid].max() >= count or ground_truth[valid].min() < 0:
                raise ValueError("人工标注类别 ID 不符合当前词表。")
            histogram = np.bincount(
                count * ground_truth[valid].astype(np.int64) + labels[valid],
                minlength=count * count,
            ).reshape(count, count)
            intersection = np.diag(histogram)
            union = histogram.sum(0) + histogram.sum(1) - intersection
            for row, hit, total in zip(rows, intersection, union):
                row["iou"] = float(hit / total) if total else None
            metrics = {
                "pixel_accuracy": float(intersection.sum() / valid.sum()),
                "mean_iou_over_nonempty_unions": float((intersection[union > 0] / union[union > 0]).mean()),
                "valid_pixels": int(valid.sum()),
                "confusion_matrix": histogram.tolist(),
                "scope": "single_image_at_input_used_resolution",
            }
    return rows, metrics, assigned, maximum, raw_argmax


def save_visualizations(directory, image, labels, assigned, rows, presence, ground_truth=None):
    count = len(rows)
    # 固定颜色由类别 ID 决定，每次测试保持一致。只把显式 background 类画成黑色。
    hues = (np.arange(count) * 0.61803398875) % 1
    from matplotlib.colors import hsv_to_rgb
    colors = (hsv_to_rgb(np.column_stack([hues, np.full(count, 0.7), np.full(count, 0.95)])) * 255).astype(np.uint8)
    for row in rows:
        if row["name"].strip().lower() == "background":
            colors[row["class_id"]] = 0
    colored = colors[labels]
    overlay = Image.blend(image.convert("RGB"), Image.fromarray(colored), OVERLAY_ALPHA)
    Image.fromarray(colored).save(directory / "segmentation_color.png")
    overlay.save(directory / "overlay.png")
    figure, axes = plt.subplots(2, 3, figsize=(18, 11))
    axes[0, 0].imshow(image)
    axes[0, 0].set_title("Input used")
    axes[0, 1].imshow(colored)
    axes[0, 1].set_title("Semantic class labels")
    axes[0, 2].imshow(overlay)
    axes[0, 2].set_title("Overlay")
    for axis in axes[0]:
        axis.axis("off")
    heat = axes[1, 0].imshow(assigned, cmap="magma", vmin=0, vmax=1)
    axes[1, 0].set_title("Score of assigned class")
    axes[1, 0].axis("off")
    figure.colorbar(heat, ax=axes[1, 0], fraction=0.046)
    shown = sorted(rows, key=lambda row: row["presence_score"], reverse=True)[:15]
    axes[1, 1].barh(
        [f'{row["class_id"]} {row["name"]}' for row in shown][::-1],
        [row["presence_score"] for row in shown][::-1],
    )
    axes[1, 1].set_xlim(0, 1)
    axes[1, 1].set_title("Top class presence scores")
    if ground_truth is not None:
        gt_rgb = np.full((*labels.shape, 3), 160, dtype=np.uint8)
        valid = ground_truth != 255
        gt_rgb[valid] = colors[ground_truth[valid]]
        axes[1, 2].imshow(gt_rgb)
        axes[1, 2].set_title("Ground truth (gray = ignored)")
        axes[1, 2].axis("off")
    else:
        present = [row for row in rows if row["pixel_count"]]
        ranked = sorted(present, key=lambda row: row["pixel_count"], reverse=True)[:15]
        axes[1, 2].barh(
            [f'{row["class_id"]} {row["name"]}' for row in ranked][::-1],
            [100 * row["pixel_fraction"] for row in ranked][::-1],
        )
        axes[1, 2].set_xlabel("Pixels (%)")
        axes[1, 2].set_title("Label coverage")
    present = [row for row in rows if row["pixel_count"]]
    handles = [
        Patch(facecolor=colors[row["class_id"]] / 255, label=f'{row["class_id"]} {row["name"]}')
        for row in present
    ]
    legend_rows = (len(handles) + 4) // 5
    legend_height = min(0.4, 0.035 * legend_rows + 0.03)
    figure.legend(handles=handles, loc="lower center", ncol=min(5, len(handles)), fontsize=8)
    figure.tight_layout(rect=(0, legend_height, 1, 1))
    figure.savefig(directory / "visualization.png", dpi=140, bbox_inches="tight")
    plt.close(figure)


def publish_run(local_directory, drive_directory, run_started=None):
    archive_started = time.perf_counter()
    manifest = {}
    for source in sorted(local_directory.iterdir()):
        if not source.is_file():
            continue
        checksum = file_sha256(source)
        temporary = drive_directory / (source.name + ".partial")
        shutil.copy2(source, temporary)
        if file_sha256(temporary) != checksum:
            raise IOError(f"Drive 文件校验失败，完整结果保留在本地。{source.name}")
        os.replace(temporary, drive_directory / source.name)
        manifest[source.name] = {"sha256": checksum, "bytes": source.stat().st_size}
    completed = {
        "status": "completed",
        "completed_at": datetime.now(ZoneInfo("Asia/Shanghai")).isoformat(),
        "files": manifest,
        "drive_archive_seconds": time.perf_counter() - archive_started,
        "total_seconds": time.perf_counter() - run_started if run_started is not None else None,
    }
    write_json(drive_directory / "COMPLETE.json.partial", completed)
    os.replace(drive_directory / "COMPLETE.json.partial", drive_directory / "COMPLETE.json")
    return manifest


def write_result_bundle(directory, original, image, labels, class_scores, presence, active, metadata, gt_raw=None):
    ground_truth = None
    if gt_raw is not None:
        gt_used = gt_raw.resize(image.size, Image.Resampling.NEAREST)
        gt_array = np.asarray(gt_used, dtype=np.int32)
        if DATASET_PRESET == "ade20k":
            if not np.isin(gt_array, np.arange(151)).all():
                raise ValueError("ADE20K 原始标注应为 0..150。请检查标注文件。")
            ground_truth = np.where(gt_array == 0, 255, gt_array - 1).astype(np.uint8)
        else:
            if not np.isin(gt_array, [*range(21), 255]).all():
                raise ValueError("VOC21 标注应为 0..20 或忽略值 255。")
            ground_truth = gt_array.astype(np.uint8)
        gt_raw.save(directory / "ground_truth_original.png")
        Image.fromarray(ground_truth).save(directory / "ground_truth_train_ids.png")
    rows, metrics, assigned, maximum, raw_argmax = summarize_arrays(
        labels, class_scores, presence, active, CLASS_INFO, ground_truth
    )
    save_visualizations(directory, image, labels, assigned, rows, presence, ground_truth)
    with Image.open(directory / "visualization.png") as figure:
        display(figure.copy())
    original.save(directory / "input_original.png")
    image.save(directory / "input_used.png")
    Image.fromarray(labels).save(directory / "labels.png")
    np.save(directory / "labels.npy", labels, allow_pickle=False)
    np.savez_compressed(
        directory / "scores.npz",
        class_scores=class_scores,
        presence_scores=presence,
        preview_admitted_class_ids=np.asarray(active, dtype=np.int32),
        predicted_class_ids=np.unique(labels).astype(np.int32),
        assigned_class_score=assigned,
        max_class_score=maximum,
        raw_argmax_labels=raw_argmax,
    )
    field_names = list(rows[0])
    with (directory / "class_results.csv").open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=field_names)
        writer.writeheader()
        for row in rows:
            writer.writerow({
                key: json.dumps(value, ensure_ascii=False) if isinstance(value, list) else value
                for key, value in row.items()
            })
    metadata.update(
        class_results=rows,
        single_image_metrics=metrics,
        predicted_class_ids=np.unique(labels).tolist(),
        preview_admitted_class_ids=np.asarray(active).tolist(),
        fraction_changed_from_argmax=float(np.mean(labels != raw_argmax)),
        array_schema={
            "class_scores": "float32 [C,H,W], 原版融合并经presence加权的类别响应，按词表类别ID排序",
            "presence_scores": "float32 [C], 预览未准入=0；二筛淘汰=首层分数；完整解码=最终分数；别名按最大值聚合",
            "assigned_class_score": "float32 [H,W], class_scores[最终标签,y,x]",
            "max_class_score": "float32 [H,W], max_c class_scores[c,y,x]",
            "raw_argmax_labels": "uint8 [H,W], 每像素响应最大的类",
            "labels": "uint8 [H,W], 原版decode返回的最终类别ID",
            "score_interpretation": "融合响应未经跨类别归一化，ECD输出可与argmax不同",
        },
    )
    shutil.copy2(vocab_path, directory / "vocabulary.txt")
    shutil.copy2(signature_path, directory / "signatures.npz")
    return rows, metrics


def run_single_image(image_path, ground_truth_path=None):
    if DATASET_PRESET != LOADED_PRESET or REPO_DIR != LOADED_REPO_DIR:
        raise RuntimeError("源码或数据集配置有变化，请重新运行第 4 步加载模型。")
    if str(WEIGHTS_PATH.resolve()) != LOADED_WEIGHT_SOURCE or BUCKET_SIZE != model.bucket_size:
        raise RuntimeError("权重路径或分桶参数有变化，请重新运行第 4 步加载模型。")
    run_id = datetime.now(ZoneInfo("Asia/Shanghai")).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
    local_directory = LOCAL_RUNS / run_id
    drive_directory = OUTPUT_ROOT / run_id
    local_directory.mkdir(parents=True, exist_ok=False)
    started = time.perf_counter()
    metadata = {
        "schema_version": 1,
        "run_id": run_id,
        "started_at": datetime.now(ZoneInfo("Asia/Shanghai")).isoformat(),
        "image_drive_path": str(image_path),
        "ground_truth_drive_path": str(ground_truth_path) if ground_truth_path else None,
        "local_directory": str(local_directory),
        "drive_directory": str(drive_directory),
        "dataset_preset": DATASET_PRESET,
        "repo_url": SOURCE_INFO["github_url"],
        "git_ref": SOURCE_INFO["git_ref"],
        "git_commit": SOURCE_INFO["git_commit"],
        "deployment_id": SOURCE_INFO["deployment_id"],
        "weight_drive_path": LOADED_WEIGHT_SOURCE,
        "weight_sha256": LOADED_WEIGHT_SHA256,
        "effective_model_config": LOADED_MODEL_CONFIG.copy(),
        "runtime": VERSION_INFO,
        "max_input_side": MAX_INPUT_SIDE,
        "overlay_alpha": OVERLAY_ALPHA,
        "seed": 0,
        "orientation_policy": "PIL RGB按存储像素方向读取，与原仓库一致",
        "prediction_id_rule": (
            "prediction 0..149 corresponds to ADE annotation 1..150; annotation 0 ignored"
            if DATASET_PRESET == "ade20k" else "prediction and annotation 0..20; 0 background; 255 ignored"
        ),
    }
    write_json(local_directory / "RUNNING.json", metadata)
    try:
        drive_directory.mkdir(parents=True, exist_ok=False)
        metadata["vocabulary_sha256"] = file_sha256(vocab_path)
        metadata["signatures_sha256"] = file_sha256(signature_path)
        write_json(local_directory / "RUNNING.json", metadata)
        write_json(drive_directory / "RUNNING.json", metadata)
        image_path = Path(image_path)
        local_source = local_directory / ("source_image" + image_path.suffix.lower())
        shutil.copy2(image_path, local_source)
        metadata["source_image_sha256"] = file_sha256(local_source)
        with Image.open(local_source) as source:
            original = source.convert("RGB")
        image = original.copy()
        if MAX_INPUT_SIDE:
            image.thumbnail((MAX_INPUT_SIDE, MAX_INPUT_SIDE), Image.Resampling.LANCZOS)
        metadata["sizes"] = {
            "original_wh": list(original.size),
            "input_used_wh": list(image.size),
            "output_wh": list(image.size),
            "scale_xy": [image.width / original.width, image.height / original.height],
            "resized": image.size != original.size,
            "model_resolution": model.processor.resolution,
            "preview_resolution": model.preview_processor.resolution,
        }
        gt_raw = None
        if ground_truth_path is not None:
            local_gt = local_directory / "source_ground_truth.png"
            shutil.copy2(ground_truth_path, local_gt)
            metadata["source_ground_truth_sha256"] = file_sha256(local_gt)
            with Image.open(local_gt) as source_gt:
                gt_raw = source_gt.copy()
            if gt_raw.size != original.size or np.asarray(gt_raw).ndim != 2:
                raise ValueError("人工标注必须是与原图同尺寸的单通道类别ID图。")

        random.seed(0)
        np.random.seed(0)
        torch.manual_seed(0)
        torch.cuda.manual_seed_all(0)
        torch.backends.cudnn.benchmark = False
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        inference_started = time.perf_counter()
        with torch.inference_mode():
            class_tensor, presence_tensor, active_tensor = model.class_maps(image, (image.height, image.width))
            label_tensor = model.decode(class_tensor, presence_tensor, active_tensor)
        torch.cuda.synchronize()
        metadata["inference_seconds"] = time.perf_counter() - inference_started
        metadata["peak_allocated_gib"] = torch.cuda.max_memory_allocated() / 1024**3
        metadata["peak_reserved_gib"] = torch.cuda.max_memory_reserved() / 1024**3
        metadata["memory_scope"] = "峰值含已加载模型；计时仅class_maps+decode，含首次运行开销"
        labels = label_tensor.cpu().numpy().astype(np.uint8)
        class_scores = class_tensor.float().cpu().numpy()
        presence = presence_tensor.float().cpu().numpy()
        active = active_tensor.cpu().numpy().astype(np.int32)
        del label_tensor, class_tensor, presence_tensor, active_tensor
        metadata["before_export_seconds"] = time.perf_counter() - started
        export_started = time.perf_counter()
        rows, metrics = write_result_bundle(
            local_directory, original, image, labels, class_scores, presence, active, metadata, gt_raw
        )
        metadata["local_export_seconds"] = time.perf_counter() - export_started
        metadata["before_drive_archive_seconds"] = time.perf_counter() - started
        write_json(local_directory / "result.json", metadata)
        publish_run(local_directory, drive_directory, started)
        print("测试与归档完成", drive_directory)
        print(f"推理 {metadata['inference_seconds']:.3f} 秒，峰值 allocated {metadata['peak_allocated_gib']:.2f} GiB")
        return {
            "run_id": run_id, "local_directory": local_directory,
            "drive_directory": drive_directory, "class_results": rows, "metrics": metrics,
        }
    except Exception as error:
        failure = {
            **metadata, "status": "failed", "error_type": type(error).__name__,
            "error": str(error), "traceback": traceback.format_exc(),
        }
        write_json(local_directory / "ERROR.json", failure)
        try:
            write_json(drive_directory / "ERROR.json", failure)
        except Exception as drive_error:
            print("Drive 写入也失败。完整错误保留在本地", local_directory, str(drive_error))
        raise


## 6. 选择图片、推理并展示

下方的 `IMAGE_RELATIVE_PATH` 留空时使用数据集的第一张图。填写相对于 `DATASET_PATH` 的路径可以换图。每次运行本格都会在 Drive 建一个独立的结果目录。

原图长边超过 `MAX_INPUT_SIDE` 时，先缩小图片以控制显存。标签和分数图以实际推理尺寸保存，运行记录包含原尺寸和缩放比例。


In [ ]:
IMAGE_RELATIVE_PATH = ""  # 例如 "images/validation/ADE_val_00000001.jpg"
GT_RELATIVE_PATH = ""     # 留空时按同名 PNG 自动查找
LOCAL_RUNS = Path("/content/activesam_runs")

image_subdir = "images/validation" if DATASET_PRESET == "ade20k" else "JPEGImages"
if IMAGE_RELATIVE_PATH:
    IMAGE_PATH = (DATASET_PATH / IMAGE_RELATIVE_PATH).resolve()
else:
    IMAGE_PATH = next(
        (p for p in sorted((DATASET_PATH / image_subdir).glob("*"))
         if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}),
        None,
    )
    if IMAGE_PATH is None:
        raise FileNotFoundError(f"{DATASET_PATH / image_subdir} 没有测试图片。")
if not IMAGE_PATH.resolve().is_relative_to(DATASET_PATH.resolve()):
    raise ValueError("测试图片须位于 DATASET_PATH 内。")
if not IMAGE_PATH.is_file():
    raise FileNotFoundError(IMAGE_PATH)

if GT_RELATIVE_PATH:
    GT_PATH = (DATASET_PATH / GT_RELATIVE_PATH).resolve()
    if not GT_PATH.is_file():
        raise FileNotFoundError(f"指定的人工标注不存在。{GT_PATH}")
else:
    gt_subdir = "annotations/validation" if DATASET_PRESET == "ade20k" else "SegmentationClass"
    candidate = DATASET_PATH / gt_subdir / f"{IMAGE_PATH.stem}.png"
    GT_PATH = candidate if candidate.is_file() else None

if GT_PATH is not None and not GT_PATH.resolve().is_relative_to(DATASET_PATH.resolve()):
    raise ValueError("人工标注须位于 DATASET_PATH 内。")
with Image.open(IMAGE_PATH) as source:
    print("输入", IMAGE_PATH, "原尺寸", source.size)
print("人工标注", GT_PATH if GT_PATH else "未提供")

LAST_RUN = run_single_image(IMAGE_PATH, GT_PATH)
display(Markdown(f"已保存到 Drive **{LAST_RUN['drive_directory']}**"))

import pandas as pd
class_table = pd.DataFrame(LAST_RUN["class_results"])
display(
    class_table.loc[class_table["pixel_count"] > 0,
                    ["class_id", "name", "presence_score", "pixel_count", "pixel_fraction",
                     "assigned_score_mean", "score_map_max"]]
    .sort_values("pixel_count", ascending=False).reset_index(drop=True)
)
if LAST_RUN["metrics"] is not None:
    metrics = LAST_RUN["metrics"]
    print("单图像素准确率", f"{metrics['pixel_accuracy']:.2%}")
    print("单图非空类别并集的平均IoU", f"{metrics['mean_iou_over_nonempty_unions']:.2%}")


## 可选 · 重新读取 Drive 结果

本格会校验归档中的文件，并从 Drive 展示叠加图。查看历史记录时，将 `run_directory` 改为对应的结果目录。


In [ ]:
from pathlib import Path
import hashlib
import json
from PIL import Image
from IPython.display import display

run_directory = LAST_RUN["drive_directory"] if "LAST_RUN" in globals() else Path("/content/drive/MyDrive/results/ActiveSAM/请填写历史结果目录")  # 可改成 Path("/content/drive/MyDrive/ActiveSAM/results/某次目录")
completed = json.loads((run_directory / "COMPLETE.json").read_text(encoding="utf-8"))
assert completed["status"] == "completed"
for name, record in completed["files"].items():
    path = run_directory / name
    with path.open("rb") as stream:
        checksum = hashlib.file_digest(stream, "sha256").hexdigest()
    if path.stat().st_size != record["bytes"] or checksum != record["sha256"]:
        raise IOError(f"结果文件校验失败。{path}")
saved_result = json.loads((run_directory / "result.json").read_text(encoding="utf-8"))
print("归档校验通过", saved_result["run_id"])
with Image.open(run_directory / "overlay.png") as overlay:
    display(overlay.copy())
# 完整数组按需读取，避免仅查看图片时占用大量内存。
# import numpy as np
# with np.load(run_directory / "scores.npz", allow_pickle=False) as scores:
#     class_scores = scores["class_scores"]  # float32 [C,H,W]
#     selected_scores = scores["assigned_class_score"]  # float32 [H,W]
#     presence_scores = scores["presence_scores"]  # float32 [C]
